In [21]:
import pandas as pd
import numpy as np

In [22]:
# Cargar archivos CSV
sales_df = pd.read_csv('/workspace/sales.csv')
inventories_df = pd.read_csv('/workspace/inventories.csv')
satisfaction_df = pd.read_csv('/workspace/satisfaction.csv')

# Mostrar las primeras filas
print("Ventas")
print(sales_df.head())

print("\nInventarios")
print(inventories_df.head())

print("\nSatisfacción")
print(satisfaction_df.head())

Ventas
   ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario Fecha_Venta
0          1  Producto A                20              100  2023-01-05
1          1  Producto B                15              200  2023-01-06
2          2  Producto A                30              100  2023-01-07
3          2  Producto C                25              300  2023-01-08
4          3  Producto A                10              100  2023-01-09

Inventarios
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización
0          1  Producto A                50          2023-01-05
1          1  Producto B                40          2023-01-06
2          2  Producto A                60          2023-01-07
3          2  Producto C                45          2023-01-08
4          3  Producto A                30          2023-01-09

Satisfacción
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
0          1                     85       2023-01-15
1          2                     90       2023-01-

In [23]:
# Eliminar valores nulos
sales_df = sales_df.dropna()
inventories_df = inventories_df.dropna()
satisfaction_df = satisfaction_df.dropna()

# Verificar información general
print(sales_df.info())
print(inventories_df.info())
print(satisfaction_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ID_Tienda         10 non-null     int64
 1   Producto          10 non-null     str  
 2   Cantidad_Vendida  10 non-null     int64
 3   Precio_Unitario   10 non-null     int64
 4   Fecha_Venta       10 non-null     str  
dtypes: int64(3), str(2)
memory usage: 532.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   ID_Tienda            10 non-null     int64
 1   Producto             10 non-null     str  
 2   Stock_Disponible     10 non-null     int64
 3   Fecha_Actualización  10 non-null     str  
dtypes: int64(2), str(2)
memory usage: 452.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column     

In [24]:
ventas_producto_tienda = sales_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum()

print("Ventas totales por producto y tienda")
print(ventas_producto_tienda)

Ventas totales por producto y tienda
ID_Tienda  Producto  
1          Producto A    20
           Producto B    15
2          Producto A    30
           Producto C    25
3          Producto A    10
           Producto B    40
4          Producto A    25
           Producto C    35
5          Producto B    20
           Producto C    30
Name: Cantidad_Vendida, dtype: int64


In [25]:
# Crear columna de ingresos
sales_df['Total_Ventas'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']

# Ingresos por tienda
ingresos_tienda = sales_df.groupby('ID_Tienda')['Total_Ventas'].sum()

print("Ingresos totales por tienda")
print(ingresos_tienda)

Ingresos totales por tienda
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Total_Ventas, dtype: int64


In [26]:
print("Resumen estadístico de ventas")
print(sales_df['Total_Ventas'].describe())

Resumen estadístico de ventas
count       10.000000
mean      5050.000000
std       3361.960407
min       1000.000000
25%       2625.000000
50%       3500.000000
75%       7875.000000
max      10500.000000
Name: Total_Ventas, dtype: float64


In [27]:
# Verificar si existe columna Categoria
if 'ID_Tienda' in sales_df.columns:
    promedio_categoria = sales_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].mean()

    print("Promedio de ventas por tienda y categoría")
    print(promedio_categoria)
else:
    print("La columna Categoria no existe en el archivo sales.csv")

Promedio de ventas por tienda y categoría
ID_Tienda  Producto  
1          Producto A    20.0
           Producto B    15.0
2          Producto A    30.0
           Producto C    25.0
3          Producto A    10.0
           Producto B    40.0
4          Producto A    25.0
           Producto C    35.0
5          Producto B    20.0
           Producto C    30.0
Name: Cantidad_Vendida, dtype: float64


In [28]:
ventas_totales_tienda = sales_df.groupby('ID_Tienda')['Total_Ventas'].sum()

print("Ventas totales por tienda")
print(ventas_totales_tienda)

Ventas totales por tienda
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Total_Ventas, dtype: int64


In [29]:
# Agrupar ventas por tienda y producto
ventas_productos = sales_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()

# Unir ventas con inventarios
inventario_ventas = pd.merge(
    inventories_df,
    ventas_productos,
    on=['ID_Tienda', 'Producto'],
    how='left'
)

# Reemplazar NaN por 0
inventario_ventas['Cantidad_Vendida'] = inventario_ventas['Cantidad_Vendida'].fillna(0)

# Calcular rotación
inventario_ventas['Rotacion_Inventario'] = (
    inventario_ventas['Cantidad_Vendida'] /
    inventario_ventas['Stock_Disponible']
)

print(inventario_ventas.head())

   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  \
0          1  Producto A                50          2023-01-05   
1          1  Producto B                40          2023-01-06   
2          2  Producto A                60          2023-01-07   
3          2  Producto C                45          2023-01-08   
4          3  Producto A                30          2023-01-09   

   Cantidad_Vendida  Rotacion_Inventario  
0                20             0.400000  
1                15             0.375000  
2                30             0.500000  
3                25             0.555556  
4                10             0.333333  


In [30]:
inventarios_criticos = inventario_ventas[
    inventario_ventas['Rotacion_Inventario'] < 0.10
]

print("Tiendas con inventarios críticos")
print(inventarios_criticos)

Tiendas con inventarios críticos
Empty DataFrame
Columns: [ID_Tienda, Producto, Stock_Disponible, Fecha_Actualización, Cantidad_Vendida, Rotacion_Inventario]
Index: []


In [31]:
rotacion_tienda = inventario_ventas.groupby('ID_Tienda')['Rotacion_Inventario'].mean()

print("Rotación promedio por tienda")
print(rotacion_tienda)

Rotación promedio por tienda
ID_Tienda
1    0.387500
2    0.527778
3    0.416667
4    0.500000
5    0.500000
Name: Rotacion_Inventario, dtype: float64


In [32]:
print("Promedio de satisfacción por tienda")

satisfaccion_tienda = satisfaction_df.groupby('ID_Tienda')['Satisfacción_Promedio'].mean()

print(satisfaccion_tienda)

Promedio de satisfacción por tienda
ID_Tienda
1    85.0
2    90.0
3    70.0
4    65.0
5    55.0
Name: Satisfacción_Promedio, dtype: float64


In [33]:
baja_satisfaccion = satisfaction_df[
    satisfaction_df['Satisfacción_Promedio'] < 60
]

print("Tiendas con baja satisfacción")
print(baja_satisfaccion)

Tiendas con baja satisfacción
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
4          5                     55       2023-01-15


In [34]:
# Ventas por tienda
ventas_por_tienda = sales_df.groupby('ID_Tienda')['Total_Ventas'].sum().reset_index()

# Satisfacción promedio
satisfaccion_promedio = satisfaction_df.groupby('ID_Tienda')['Satisfacción_Promedio'].mean().reset_index()

# Unir información
analisis_general = pd.merge(
    ventas_por_tienda,
    satisfaccion_promedio,
    on='ID_Tienda'
)

print("Relación entre ventas y satisfacción")
print(analisis_general)

Relación entre ventas y satisfacción
   ID_Tienda  Total_Ventas  Satisfacción_Promedio
0          1          5000                   85.0
1          2         10500                   90.0
2          3          9000                   70.0
3          4         13000                   65.0
4          5         13000                   55.0


In [35]:
ventas_array = sales_df['Total_Ventas'].to_numpy()

print(ventas_array)

[ 2000  3000  3000  7500  1000  8000 10500  2500  4000  9000]


In [36]:
mediana_ventas = np.median(ventas_array)

print("Mediana de ventas:")
print(mediana_ventas)

Mediana de ventas:
3500.0


In [37]:
desviacion_estandar = np.std(ventas_array)

print("Desviación estándar:")
print(desviacion_estandar)

Desviación estándar:
3189.4356867634124


In [39]:
# Semilla para reproducibilidad
np.random.seed(42)

# Simular ventas futuras
proyeccion_ventas = np.random.normal(
    loc=np.mean(ventas_array),
    scale=np.std(ventas_array),
    size=12
)

print("Proyección de ventas futuras")
print(proyeccion_ventas)
# Estadísticas de las proyecciones futuras
media_proyeccion = np.mean(proyeccion_ventas)
mediana_proyeccion = np.median(proyeccion_ventas)
desviacion_proyeccion = np.std(proyeccion_ventas)

print("\nEstadísticas de las proyecciones futuras")

print("Media de proyección:")
print(media_proyeccion)

print("Mediana de proyección:")
print(mediana_proyeccion)

print("Desviación estándar de proyección:")
print(desviacion_proyeccion)

Proyección de ventas futuras
[ 6634.23784573  4609.01490364  7115.76093733  9907.60577603
  4303.18287048  4303.23523392 10086.79771077  7497.68371242
  3552.64163948  6780.46036522  3571.95907267  3564.58490358]

Estadísticas de las proyecciones futuras
Media de proyección:
5993.930414272573
Mediana de proyección:
5621.6263746868535
Desviación estándar de proyección:
2272.6534108808137
